In [28]:
from typing import Dict, List, Set, NamedTuple

In [2]:
Edge = NamedTuple('Edge', [('v', int),
                           ('w', int)])

In [3]:
edges = [[Edge(0,1),Edge(1,2), Edge(2,3), Edge(2,4)], [Edge(0,5)]]

In [8]:
from itertools import chain

In [55]:
def obtain_predecessor_successor(edges: List[List[Edge]]):
    predecessor = {}
    successor = {i:[] for i in get_vertices(edges)}
    for e in chain.from_iterable(edges):
        predecessor[e.w] = e.v
        successor[e.v].append(e.w)
    return predecessor, successor

In [56]:
obtain_predecessor_successor(edges)

({1: 0, 2: 1, 3: 2, 4: 2, 5: 0},
 {0: [1, 5], 1: [2], 2: [3, 4], 3: [], 4: [], 5: []})

In [57]:
def obtain_leaves(edges: List[List[Edge]]):
    _, successor = obtain_predecessor_successor(edges)
    leaves = []
    for v in successor:
        if successor[v] == []:
            leaves.append(v)
    return leaves

In [29]:
def get_vertices(edge: List[List[Edge]]) -> Set[int]:
    return set(chain.from_iterable(e for e in chain.from_iterable(edges)))

In [65]:
def count_successors(vertex, successors):
    i = 1
    for s in successors[vertex]:
        i+= count_successors(s, successors)
    return i

def compute_capacities(edges):
    _, successor = obtain_predecessor_successor(edges)
    return {i:count_successors(i, successor) for i in get_vertices(edges)}

In [67]:
compute_capacities(edges)

{0: 6, 1: 4, 2: 3, 3: 1, 4: 1, 5: 1}

In [22]:
import numpy as np

In [25]:
CABLE_COSTS = np.array([0.58,0.87,1.24,1.95,3.13,5.19,6.9])
CABLE_LENGTHS = np.array([38.36,47.08,56.04,69.3,84.87,102.79,120.31])
CABLE_CAPACITIES = np.array([56,73,92,124,162,209,250])

In [121]:
coords = np.random.rand(6,2) * 150

In [122]:
from scipy.spatial.distance import cdist

In [123]:
def assign_power_cables(edges: List[List[Edge]], coordinates, cable_lengths, cable_capacities) -> Dict[Edge,int]:
    capacities = compute_capacities(edges)
    distances = cdist(coordinates, coordinates)
    
    assignment = {}
    
    for e in chain.from_iterable(edges):
        length = distances[e.v,e.w] <= cable_lengths
        capacity = capacities[e.v] <= cable_capacities
        
        best_cable_index = np.where(np.logical_and(length,capacity) == True)
        if best_cable_index[0].size == 0:
            raise RuntimeError('No suitable cable found')
            
        assignment[e] = best_cable_index[0][0]
    return assignment

In [124]:
assign_power_cables(edges, coords, CABLE_LENGTHS, CABLE_CAPACITIES)

{Edge(v=0, w=1): 4,
 Edge(v=0, w=5): 6,
 Edge(v=1, w=2): 3,
 Edge(v=2, w=3): 5,
 Edge(v=2, w=4): 0}